# DrugBAN on DAVIS — the current-generation subject

The audit's three subjects are from 2020-2022, and the first question a reviewer asks is
whether its verdict binds on what people build now. **DrugBAN** (Bai et al., *Nature
Machine Intelligence* 2023) is the answer: its title claims interpretability, it reports
cross-domain generalisation, and its code is maintained and MIT-licensed.

This notebook trains its **12 DAVIS cells** — four levels, three seeds — on two T4s.
Nothing else: KIBA, the ladders and the faithfulness runs happen elsewhere.

| | |
|---|---|
| model | DrugBAN, their architecture from their own `configs.py`, unmodified |
| recipe | Adam 5e-5, batch 64, up to 100 epochs — **theirs** |
| what we change | early stopping on validation loss (patience 15, floor 10) and `BCEWithLogitsLoss` on their single logit, so this cell is selected and scored like every other cell in the audit; domain adaptation off, their own default for in-domain runs |
| cost | ~0.3-0.7 h per cell at the DAVIS median, so **12 cells ≈ 2-4 h per GPU**: one commit |

**Why it needs an install step the other notebooks do not.** DrugBAN featurises a drug as
a DGL graph. DGL is not on Kaggle's image and its wheels are pinned to a torch/CUDA
build, so section 4 installs it and then *proves* it works by running the real model
forward once. If that check fails, stop: every cell would fail the same way an hour later.

## What to do

1. **Settings** (right panel): Accelerator **GPU T4 x2**, Internet **On**.
2. Run all. Nothing to edit unless you are resuming — see section 6.
3. **Save Version -> Save & Run All (Commit)**.


## 1. Settings — normally nothing to change

In [ ]:
# ============================================================================
# SETTINGS
# ============================================================================

RESTORE_FROM = None      # second commit onward: '/kaggle/input' (searches every input)

# ============================================================================
# Everything below is the plan. Read it; do not edit it.
# ============================================================================

DATASET = 'davis'
TASK = 'binary'
BRANCH = 'main'
MODEL = 'drugban'
LEVELS = ['random', 'cold_drug', 'cold_target', 'cold_pair']
SEEDS = [1, 2, 3]

# Their SOLVER block (baselines/DrugBAN/configs.py): batch 64, lr 5e-5, 100 epochs.
# Passed explicitly so this notebook's log records what was trained rather than
# whatever a future edit to their config would silently change.
BATCH_SIZE = 64
LR = 5e-5
EPOCHS = 100

# Full precision. DrugBAN has not been validated under --amp here, and the one model in
# this audit that was left unvalidated under float16 (MolTrans on KIBA) produced NaN from
# batch ~4,040 of epoch 1. Twelve cells at ~0.5 h do not need the speed-up badly enough
# to risk a silent divergence.
AMP = False

# 12 cells over two GPUs, most expensive first. cold_pair is the cheapest (15,190 train
# rows against 21,658), so it goes last on both queues; the two queues are equal in rows.
CELLS = [(level, seed) for level in LEVELS for seed in SEEDS]
QUEUE0 = [c for i, c in enumerate(CELLS) if i % 2 == 0]
QUEUE1 = [c for i, c in enumerate(CELLS) if i % 2 == 1]
TOTAL_CELLS = len(CELLS)

print(f'{TOTAL_CELLS} cells: {MODEL} x {len(LEVELS)} levels x {len(SEEDS)} seeds')
print('GPU 0:', QUEUE0)
print('GPU 1:', QUEUE1)
print(f'precision: {"mixed" if AMP else "full (fp32)"} | batch {BATCH_SIZE} | lr {LR}')


## 2. Check the GPU(s)

In [ ]:
import time
START = time.time()          # the 11-hour self-stop is measured from here

import torch

assert torch.cuda.is_available(), 'No CUDA. Settings -> Accelerator -> GPU.'
N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')
print('torch  :', torch.__version__)
print('cuda   :', torch.version.cuda)

assert N_GPU >= 2, (
    f'Kaggle gave {N_GPU} GPU(s). Settings -> Accelerator -> GPU T4 x2. On one GPU this '
    'notebook still finishes, but in twice the wall time -- re-run this cell after '
    'switching, or accept it knowingly.')


## 3. Clone the repo

DrugBAN itself is vendored in the repo (`baselines/DrugBAN`, MIT, unmodified), so there
is nothing to fetch from GitHub for the model.

In [ ]:
import os

REPO = 'https://github.com/Mahim56207/ColdSite-DTI_New.git'
WORK = '/kaggle/working'
SRC  = f'{WORK}/ColdSite-DTI_New'

if not os.path.exists(SRC):
    !git clone --branch {BRANCH} {REPO} {SRC}
os.chdir(SRC)
!git checkout {BRANCH}
!git pull origin {BRANCH}
!pip install -q tabulate

for _needed in ('src/model/train_drugban.py', 'src/evaluation/drugban_adapter.py',
                'baselines/DrugBAN/models.py', 'src/model/resume.py'):
    assert os.path.exists(_needed), (
        f'{_needed} is missing from branch {BRANCH!r}. Push the DrugBAN commit to '
        'origin/main before running this notebook, then re-run this cell.')

RESULTS = f'{WORK}/results'
os.makedirs(RESULTS, exist_ok=True)
print()
!git log --oneline -1
print('results ->', RESULTS)


## 4. Install DGL — and prove it works

DGL's wheels are built against one torch/CUDA pair, so the index URL is derived from the
torch Kaggle actually gave us rather than hard-coded. `torchdata` must be 0.7.x: DGL 2.x
imports `torchdata.datapipes`, which later versions removed.

The last cell here builds the real DrugBAN and runs one forward pass. A green tick means
the model, the featuriser and the graph library agree; anything else means stop.

In [ ]:
import os, re, subprocess, sys, torch

torch_mm = '.'.join(torch.__version__.split('.')[:2])          # e.g. '2.6'
cuda_tag = 'cu' + (torch.version.cuda or '121').replace('.', '')
urls = [f'https://data.dgl.ai/wheels/torch-{torch_mm}/{cuda_tag}/repo.html',
        f'https://data.dgl.ai/wheels/torch-{torch_mm}/repo.html',
        'https://data.dgl.ai/wheels/repo.html']

!pip install -q "torchdata==0.7.1" dgllife rdkit yacs prettytable

installed = False
for url in urls:
    print(f'-> trying {url}')
    code_ = subprocess.call([sys.executable, '-m', 'pip', 'install', '-q', 'dgl', '-f', url])
    if code_ == 0:
        try:
            subprocess.check_call([sys.executable, '-c', 'import dgl'],
                                  env={**os.environ, 'DGLBACKEND': 'pytorch'})
            installed = True
            print(f'   installed from {url}')
            break
        except subprocess.CalledProcessError:
            print('   installed but will not import; trying the next index')
assert installed, (
    'DGL would not install for this torch/CUDA build. Pin the environment to a version '
    'that worked (Settings -> Environment -> pin to original), or install a matching '
    'torch first. Every DrugBAN cell would fail without it.')

os.environ['DGLBACKEND'] = 'pytorch'


In [ ]:
# The proof: build their model and run one real pair through it.
import os
os.environ['DGLBACKEND'] = 'pytorch'

import dgl, torch
from src.evaluation.drugban_adapter import DrugBANAdapter, _config
import sys
sys.path.insert(0, 'baselines/DrugBAN')
from models import DrugBAN

seq = ('MKKFFDSRREQGGSGLGSGSSGGGGSTSGLGSGYIGRVFGIGRQQVTVDEVLAEGGFAIVFLVRTSNGMKCALKRMF'
       'VNNEHDLQVCKREIQIMRDLSGHKNIVGYIDSSINNVSSGDVWEVLILM')
graph, protein = DrugBANAdapter.encode('CC(=O)Oc1ccccc1C(=O)O', seq)
model = DrugBAN(**_config()).eval()
with torch.no_grad():
    _v_d, _v_p, score, att = model(dgl.batch([graph]), protein.unsqueeze(0), mode='eval')
print('dgl', dgl.__version__, '| score', float(score.reshape(-1)[0]))
print('attention map', tuple(att.shape), '= (batch, heads, drug atoms, protein positions)')
assert att.shape[1] == 2 and att.shape[2] == 290, att.shape
print('\nDGL, the featuriser and DrugBAN all agree. Safe to train.')


## 5. Build the splits — and verify they match

In [ ]:
!python -m src.data.build_splits 2>&1 | grep -E 'davis|leakage'

import pandas as pd
from src.model.dataset import BINARY_THRESHOLD

# Recorded from data/splits/davis on the Mac, 2026-09-18. A split that does not match
# these numbers is not the split the other three models were trained on, and the whole
# point of adding this model is that it faces the identical test.
EXPECTED = {
    'random':      (21039, 3006, 6011),
    'cold_drug':   (21658, 2652, 5746),
    'cold_target': (21080, 2992, 5984),
    'cold_pair':   (15190,  264, 1144),
}
assert BINARY_THRESHOLD['davis'] == 7.0, BINARY_THRESHOLD
print(f"binary threshold: DAVIS pKd >= {BINARY_THRESHOLD['davis']}")

ok = True
for level, expected in EXPECTED.items():
    sizes = tuple(len(pd.read_csv(f'data/splits/{DATASET}/{level}/{part}.csv'))
                  for part in ('train', 'valid', 'test'))
    mark = 'OK' if sizes == expected else f'MISMATCH, expected {expected}'
    ok &= sizes == expected
    print(f'{level:12s} {str(sizes):28s} {mark}')
assert ok, 'splits differ from the recorded ones -- stop and find out why'
print('\nsplits match the record.')


## 6. Restore from a previous commit

Only needed if a commit was cut short. Set `RESTORE_FROM = '/kaggle/input'` in section 1,
having attached this account's own output as a private dataset. Finished cells are then
skipped and an interrupted one continues from its last finished epoch.

In [ ]:
import shutil, glob

if RESTORE_FROM:
    assert os.path.isdir(RESTORE_FROM), f'not a directory: {RESTORE_FROM}'
    copied = skipped = 0
    for src in glob.glob(f'{RESTORE_FROM}/**/*', recursive=True):
        name = os.path.basename(src)
        if not name.endswith(('.pt', '_results.json', '_history.json')):
            continue
        dst = os.path.join(RESULTS, name)
        if os.path.exists(dst):
            skipped += 1
            continue
        shutil.copy2(src, dst)
        copied += 1
    print(f'restored {copied} file(s), left {skipped} already present')
else:
    print('RESTORE_FROM is None -- starting from an empty results folder.')


## 7. The runner

Two queues, one per GPU, each cell a separate process so a crash cannot take the other
GPU with it. A STATUS line every 10 minutes carries the measured minutes-per-epoch and
when each queue expects to finish. The whole thing self-stops at 11 hours, an hour inside
Kaggle's limit, so the commit has time to save its output.

In [ ]:
import glob, json, os, re, subprocess, threading, time
from src.model.checkpoint_naming import checkpoint_path, results_path, run_tag

DEADLINE = START + 11 * 3600
STATUS_EVERY = 600          # seconds between STATUS lines

QUEUE_OF = {'0': QUEUE0, '1': QUEUE1}
if N_GPU < 2:                      # one GPU: the same work, one queue, twice the wall time
    QUEUE_OF = {'0': QUEUE0 + QUEUE1}
    print('WARNING: one GPU. Running all cells in a single queue.')

MODEL_NAME = {'src.model.train_drugban': 'DrugBAN'}
MODEL_KEY = {'src.model.train_drugban': 'drugban'}

# A projection, not a measurement: DrugBAN has never been timed on a T4. The STATUS line
# prints the measured rate beside it with their ratio, so the first cell replaces the
# guess with a fact -- if the drift factor is far from 1, believe the measurement.
MIN_PER_EPOCH = {'drugban': 0.6}
EPOCHS_MIN, EPOCHS_TYPICAL = 25, 36        # the early-stopping floor, and the DAVIS median
HEADER = re.compile(r'^\w+/(\w+)/seed(\d+)\s*$')          # run_grid: a cell starts
SKIPPED = re.compile(r'^\[skip\] \w+/(\w+)/seed(\d+)')     # run_grid: a cell was done
EPOCH = re.compile(r'^\s*(?:epoch|Epoch)\s+(\d+)')
SAVED = re.compile(r'Saved -> (\S+_results\.json)')


def hours_left():
    return (DEADLINE - time.time()) / 3600


def note_epoch(w, number, now):
    """Record an epoch boundary and keep a mean seconds-per-epoch.

    Called for every line that looks like an epoch header. The baselines print
    "epoch 1 train batch 7/10 ..." for every batch, so the same number arrives many
    times and only a CHANGE is a boundary. The first epoch of a cell also carries the
    split encoding (KIBA is 83,000 rows), so it is the starting mark but its own
    duration is never counted in the rate.
    """
    if number != w['last_epoch']:
        gap = number - w['last_epoch'] if w['last_epoch'] else 0
        if gap > 0 and w['last_epoch_at'] is not None:
            # Divide by the gap: a resumed cell's first reported epoch is not 1, and one
            # elapsed stretch may cover several epochs. Weighted mean, so a stretch of
            # three epochs counts three times as much as a single one.
            per = (now - w['last_epoch_at']) / gap
            n = w['timed_epochs']
            w['sec_per_epoch'] = (per if not n
                                  else (w['sec_per_epoch'] * n + per * gap) / (n + gap))
            w['timed_epochs'] = n + gap
        w['last_epoch'], w['last_epoch_at'] = number, now
    w['epoch'] = str(number)
    return w


def eta(w, now, deadline):
    """Two lines about one GPU: the measured rate, and when it finishes.

    `w` is that GPU's live state. Returns [] until a rate exists -- claiming an ETA from
    a single epoch would mean quoting the split-encoding time as the epoch time.
    """
    rate = w.get('sec_per_epoch')
    key, index = w.get('key'), w.get('cell', 0)
    if not rate or not key:
        return []
    predicted = MIN_PER_EPOCH[key] * 60
    drift = rate / predicted if predicted else float('nan')
    done = int(w.get('epoch') or 0)
    lines = [f"      {rate / 60:.1f} min/epoch measured, {predicted / 60:.1f} projected "
             f"(x{drift:.2f}) over {w.get('timed_epochs', 0)} epoch(s)"]

    # this cell, if it stops at the floor or at the DAVIS median
    at = []
    for label, total in (('min', EPOCHS_MIN), ('median', EPOCHS_TYPICAL)):
        left = max(total - done, 0) * rate
        at.append(f"{label} {time.strftime('%H:%M', time.localtime(now + left))}"
                  f" (+{left / 3600:.1f} h)")
    lines.append(f"      this cell ends: {' | '.join(at)}")

    # the rest of this GPU's queue, rescaled by the drift we are actually seeing
    queued = ORDERED.get(str(w.get('gpu')), [])
    remaining = queued[index:]                      # cells not started yet
    if remaining:
        rest = sum(HOURS[m][0] for m, _lv, _s in remaining) * drift
        this_cell = max(EPOCHS_TYPICAL - done, 0) * rate / 3600
        total_left = rest + this_cell
        verdict = ('inside this commit' if now + total_left * 3600 <= deadline
                   else f'needs about {int(total_left / 11) + 1} more commit(s)')
        lines.append(f"      queue: {len(remaining)} cell(s) after this one, "
                     f"~{total_left:.1f} h left at the median -> {verdict}")
    else:
        this_cell = max(EPOCHS_TYPICAL - done, 0) * rate / 3600
        verdict = ('inside this commit' if now + this_cell * 3600 <= deadline
                   else 'needs another commit')
        lines.append(f"      last cell of this queue, ~{this_cell:.1f} h left at the "
                     f"median -> {verdict}")
    return lines


def cells_done():
    """This account's finished cells. Counting a MODELS x LEVELS x SEEDS product would
    count cells another account owns and report progress that is not ours."""
    done = 0
    for queue in MY_CELLS.values():
        for model, split, seed in queue:
            tag = run_tag(DATASET, split, TASK, seed)
            if os.path.exists(results_path(RESULTS, tag, model=model)):
                done += 1
    return done


def same_as_other_seed(results_file):
    """Another seed of this model and split with exactly the same test metrics means the
    seed never reached training -- MolTrans's vendored import reseeded torch with 1, so
    its three DAVIS seeds were one run three times (2026-09-13)."""
    try:
        mine = json.load(open(results_file))['test_metrics']
        for other in glob.glob(re.sub(r'_seed\d+', '_seed[0-9]', results_file)):
            if other != results_file and json.load(open(other))['test_metrics'] == mine:
                return (f'SEEDS IDENTICAL: same test metrics as {os.path.basename(other)} '
                        f'-- is --seed reaching training? Stop and check before going on.')
    except Exception:
        return None
    return None

def _arg(cmd, flag):
    return cmd[cmd.index(flag) + 1] if flag in cmd else '-'


def _cells_in(cmd):
    if cmd[3] == 'src.model.run_grid':
        return len(_arg(cmd, '--splits').split(',')) * len(_arg(cmd, '--seeds').split(','))
    return 1


def run_parallel(queues, label):
    """queues: {gpu: [command, ...]}. Returns True if the deadline cut it short."""
    current, state = {}, {'deadline': False}
    where = {g: {'model': '-', 'split': '-', 'seed': '-', 'epoch': '-', 'cell': 0,
                 'total': sum(_cells_in(c) for c in q), 'gpu': g, 'key': None,
                 'sec_per_epoch': None, 'timed_epochs': 0, 'last_epoch': None,
                 'last_epoch_at': None}
             for g, q in queues.items()}

    def tag(g):
        w = where[g]
        return f"[GPU{g} {w['model']} {w['split']} s{w['seed']} · cell {w['cell']}/{w['total']}]"

    def begin_cell(g, split, seed):
        where[g].update(split=split, seed=seed, epoch='-', sec_per_epoch=None,
                        timed_epochs=0, last_epoch=None, last_epoch_at=None)
        where[g]['cell'] += 1

    def worker(gpu, commands):
        env = {**os.environ, 'CUDA_VISIBLE_DEVICES': gpu, 'PYTHONUNBUFFERED': '1'}
        with open(f'{WORK}/{label}_gpu{gpu}.log', 'a') as log:
            for cmd in commands:
                if time.time() > DEADLINE:
                    return
                module = cmd[3]
                where[gpu]['model'] = MODEL_NAME.get(module, module)
                where[gpu]['key'] = MODEL_KEY.get(module)
                if module != 'src.model.run_grid':          # one command = one cell
                    begin_cell(gpu, _arg(cmd, '--split'), _arg(cmd, '--seed'))
                proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True, bufsize=1)
                current[gpu] = proc
                for line in proc.stdout:
                    if module == 'src.model.run_grid':      # follow run_grid's own cells
                        m = HEADER.match(line) or SKIPPED.match(line)
                        if m:
                            begin_cell(gpu, m.group(1), m.group(2))
                    m = EPOCH.match(line)
                    if m:
                        note_epoch(where[gpu], int(m.group(1)), time.time())
                    print(f'{tag(gpu)} {line}', end='', flush=True)
                    log.write(line)
                    log.flush()
                    m = SAVED.search(line)
                    if m:
                        try:
                            auc = json.load(open(m.group(1)))['test_metrics'].get('auroc')
                            auc = f'{auc:.4f}'
                        except Exception:
                            auc = '?'
                        print(f'  ✓ {tag(gpu)} finished -- test AUROC {auc}   '
                              f'[{cells_done()}/{TOTAL_CELLS} cells complete]', flush=True)
                        warning = same_as_other_seed(m.group(1))
                        if warning:
                            print(f'  !! {tag(gpu)} {warning}', flush=True)
                code_ = proc.wait()
                if code_ != 0 and not state['deadline']:
                    print(f'{tag(gpu)} !! exited {code_}: {" ".join(cmd[3:])}', flush=True)

    threads = [threading.Thread(target=worker, args=(g, q), daemon=True)
               for g, q in queues.items()]
    for t in threads:
        t.start()
    last_status = 0.0
    while any(t.is_alive() for t in threads):
        if time.time() - last_status >= STATUS_EVERY:
            last_status = time.time()
            now_ = time.time()
            print(f"\n=== STATUS {time.strftime('%H:%M')} | "
                  f"{cells_done()}/{TOTAL_CELLS} cells complete | "
                  f"{(now_ - START) / 3600:.1f} h in, {hours_left():.1f} h before the "
                  f"stop ===", flush=True)
            for g, w in sorted(where.items()):
                print(f"   GPU{g}: {w['model']} {w['split']} s{w['seed']} "
                      f"cell {w['cell']}/{w['total']} epoch {w['epoch']}", flush=True)
                for line in eta(w, now_, DEADLINE):
                    print(line, flush=True)
            print(flush=True)
        if time.time() > DEADLINE and not state['deadline']:
            state['deadline'] = True
            print('\n*** 11-hour mark: stopping so this commit can save its output. '
                  'Unfinished cells continue from their last finished epoch next commit. ***\n', flush=True)
            for proc in list(current.values()):
                if proc.poll() is None:
                    proc.terminate()
        time.sleep(15)
    return state['deadline']


def drugban_cmd(level, seed):
    """One cell. Their recipe (batch, lr, epochs) comes from section 1; the early
    stopping and the patience are the audit's, identical to every other subject."""
    return ['python', '-u', '-m', 'src.model.train_drugban',
            '--split-dir', f'data/splits/{DATASET}/{level}', '--dataset', DATASET,
            '--split', level, '--seed', str(seed),
            '--batch-size', str(BATCH_SIZE), '--lr', str(LR),
            '--min-epochs', '10', '--epochs', str(EPOCHS), '--patience', '15',
            '--checkpoint-dir', RESULTS, '--results-dir', RESULTS,
            '--skip-if-done'] + (['--amp'] if AMP else [])


# Largest split first in each queue: it is the cell most likely to meet the 11-hour stop,
# and the one whose resume file is better written early than late.
TRAIN_ROWS = {'random': 21039, 'cold_drug': 21658, 'cold_target': 21080, 'cold_pair': 15190}

# (high, low) hours per cell, the shape `eta` reads: 36 and 25 epochs at the projected
# rate above. Replaced in practice by the measured rate once the first cell reports.
HOURS = {'drugban': (EPOCHS_TYPICAL * MIN_PER_EPOCH['drugban'] / 60,
                     EPOCHS_MIN * MIN_PER_EPOCH['drugban'] / 60)}

# Triples, because `eta` and `cells_done` both unpack (model, level, seed).
ORDERED = {gpu: [(MODEL, level, seed)
                 for level, seed in sorted(cells, key=lambda c: -TRAIN_ROWS[c[0]])]
           for gpu, cells in sorted(QUEUE_OF.items())}
QUEUES = {gpu: [drugban_cmd(level, seed) for _model, level, seed in cells]
          for gpu, cells in sorted(ORDERED.items())}
MY_CELLS = ORDERED

for gpu, cells in sorted(ORDERED.items()):
    print(f'GPU {gpu}: in this order')
    for _model, level, seed in cells:
        print(f'   drugban  {level:12s} seed {seed}   {TRAIN_ROWS[level]:,} train rows  '
              f'~{HOURS["drugban"][1]:.1f}-{HOURS["drugban"][0]:.1f} h')
print(f'{hours_left():.1f} h left before the self-stop')


## 8. Launch

In [ ]:
if hours_left() < 0.5:
    raise SystemExit('less than 30 minutes before the self-stop -- not worth starting')

cut = run_parallel(QUEUES, f'drugban_{DATASET}')
print()
print(f'{cells_done()}/{TOTAL_CELLS} cells complete')
if cut:
    print('CUT SHORT by the 11-hour stop. Download the output, make it a dataset, attach '
          "it, set RESTORE_FROM = '/kaggle/input' in section 1, and run again: finished "
          'cells are skipped and an interrupted cell continues from its last epoch.')
else:
    print('every cell finished')


## 9. What landed

One row per cell. `AUROC` should be believable for DAVIS: ~0.85-0.95 at random, lower at
the cold levels. A value at 0.5 means the cell never learned; above 0.98 means look for
leakage before celebrating.

In [ ]:
import glob, json

rows = []
for path in sorted(glob.glob(f'{RESULTS}/*_{MODEL}_results.json')):
    r = json.load(open(path))
    rows.append((r['split'], r['seed'], r['test_metrics']['auroc'],
                 r['test_metrics']['auprc'], r['best_epoch'],
                 r.get('resumed_after_epoch') or '-'))

print(f'{"level":12s} {"seed":>4s} {"AUROC":>7s} {"AUPRC":>7s} {"best":>5s} {"resumed":>8s}')
for level, seed, auroc, auprc, best, resumed in sorted(rows):
    print(f'{level:12s} {seed:>4} {auroc:>7.4f} {auprc:>7.4f} {best:>5} {str(resumed):>8s}')
print(f'\n{len(rows)}/{TOTAL_CELLS} cells')

seen = {}
for level, seed, auroc, *_ in rows:
    if (level, round(auroc, 6)) in seen:
        print(f'!! {level} seed {seed} has the same AUROC as seed {seen[(level, round(auroc, 6))]}'
              ' -- is --seed reaching training? Do not merge these.')
    seen[(level, round(auroc, 6))] = seed


## 10. Take the results with you

`drugban_davis_results.zip` is what the analysis needs. Download it from the Output panel.
Its checkpoints are small (a few MB each), unlike MolTrans's.

In [ ]:
import subprocess

RES_ZIP = f'{WORK}/drugban_{DATASET}_results.zip'
finished = [p for p in glob.glob(f'{RESULTS}/*')
            if not os.path.basename(p).endswith('_resume.pt')]
resumes = glob.glob(f'{RESULTS}/*_resume.pt')

if os.path.exists(RES_ZIP):
    os.remove(RES_ZIP)
subprocess.run(['zip', '-q', '-j', RES_ZIP, *finished], check=True)
print(f'{os.path.basename(RES_ZIP)}: {len(finished)} file(s), '
      f'{os.path.getsize(RES_ZIP)/1e6:.0f} MB')

if resumes:
    RESUME_ZIP = f'{WORK}/drugban_{DATASET}_resume.zip'
    if os.path.exists(RESUME_ZIP):
        os.remove(RESUME_ZIP)
    subprocess.run(['zip', '-q', '-j', RESUME_ZIP, *resumes], check=True)
    print(f'{os.path.basename(RESUME_ZIP)}: {len(resumes)} unfinished cell(s) -- upload '
          'this one too if you need another commit.')
